# FLEO-FER on Kaggle GPU

Train the FLEO-augmented YOLOv12-S emotion detector on FER2013 + RAF-DB, compute the fold-out delta, and export R1/R2/R3 ONNX for Vitis AI / ZCU104.

**Settings (right sidebar):** Accelerator = GPU (P100 or T4), Internet = ON.  
**Add Input:** search & attach `msambare/fer2013` **and** `shuvoalok/raf-db-dataset`.

## 1. Clone + install (do NOT reinstall torch — Kaggle ships CUDA torch)

In [ ]:
!git clone https://github.com/olfa-askri/FLEO.git
%cd FLEO
!pip install -q ultralytics onnx onnxruntime onnxscript
import torch; print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Inspect what is actually mounted
Run this first — it prints the real dataset folder names under `/kaggle/input` so you can pass the correct `--src`. If a dataset is missing here, attach it via **Add Input**.

In [ ]:
import os
for d in sorted(os.listdir('/kaggle/input')):
    print('/kaggle/input/' + d)
    sub = os.path.join('/kaggle/input', d)
    for x in sorted(os.listdir(sub))[:8]:
        print('   ', x)

## 3. Prepare datasets → YOLO detection format
The prep auto-discovers images under `--src` (FER emotion-folders; RAF numeric/emotion folders or `*labels*.csv`), so nesting doesn't matter. Adjust the two paths to match the names printed in step 2 if they differ.

In [ ]:
!python -m data.prepare_fer2013 --src /kaggle/input/fer2013        --out datasets/fer2013
!python -m data.prepare_rafdb   --src /kaggle/input/raf-db-dataset --out datasets/rafdb
print('---'); import os
for n in ['fer2013','rafdb']:
    p=f'datasets/{n}/images'
    if os.path.isdir(p):
        print(n, {s: len(os.listdir(f'{p}/{s}')) for s in os.listdir(p)})

## 4. 1-minute smoke test on synthetic data (validates the full pipeline before spending quota)

In [ ]:
!python -m data.make_synthetic --out datasets/synthetic
!python -m scripts.train --data datasets/synthetic/data.yaml --variant fleo --epochs 2 --imgsz 96 --batch 8 --device 0 --workers 2

## 5. Full matrix per dataset (start with 1 seed; use `--seeds 0 1 2` for the paper's mean±sd)

In [ ]:
!python -m scripts.run_matrix --data datasets/fer2013/data.yaml --dataset fer2013 \n    --seeds 0 --epochs 100 --imgsz 160 --batch 64 --device 0

In [ ]:
!python -m scripts.run_matrix --data datasets/rafdb/data.yaml --dataset rafdb \n    --seeds 0 --epochs 100 --imgsz 160 --batch 64 --device 0

## 6. Export the three deployment routes (FP32 ONNX for Vitis AI)

In [ ]:
W='runs/fleo/fleo_seed0/weights/best.pt'
!python -m scripts.export --weights {W} --route r1 --imgsz 160 --verify
!python -m scripts.export --weights {W} --route r2 --imgsz 160 --verify
!python -m scripts.export --weights {W} --route r3 --imgsz 160
!ls -la export

## 7. Bundle artifacts to download (Output tab) or Save Version to persist

In [ ]:
!cd /kaggle/working/FLEO && zip -qr /kaggle/working/fleo_artifacts.zip export results runs/fleo/*/weights/best.pt 2>/dev/null; echo 'done -> /kaggle/working/fleo_artifacts.zip'